# Second-Hand Car Price Prediction (Linear Regression)

Predicts used-car prices from features such as brand, mileage, engine size, year, and registration status, using multiple linear regression.

**Data:** `1.04. Real-life example.csv` (used-car listings)

**Approach:** clean data → remove outliers → check OLS assumptions → log-transform the target → check multicollinearity → encode categoricals → train/test a linear regression → evaluate predictions.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import seaborn as sns
sns.set()

## 2. Load the data

In [ ]:
raw_data = pd.read_csv("1.04. Real-life example.csv")

In [ ]:
raw_data.describe(include="all")


## 3. Clean the data
Drop the `Model` column (too many unique values to be useful as-is) and remove rows with missing values.

In [ ]:
data = raw_data.drop(['Model'],axis=1)

In [ ]:
data.isnull().sum()


In [ ]:
data_no_mv=data.dropna(axis=0)

## 4. Remove outliers
Inspect the distribution of each numeric feature and trim the extreme tail (top/bottom ~1%) so a few outliers don't distort the regression.

**Price**

In [ ]:
sns.distplot(data_no_mv['Price'])

In [ ]:
q=data_no_mv['Price'].quantile(0.99)
data_1=data_no_mv[data_no_mv['Price']<q]
data_1

In [ ]:
sns.displot(data_1['Price'])

**Mileage**

In [ ]:
sns.distplot(data_no_mv['Mileage'])

In [ ]:
q=data_1['Mileage'].quantile(0.99)
data_2=data_1[data_1['Mileage']<q]
data_2

In [ ]:
sns.distplot(data_2['Mileage'])

**Engine volume** (drop implausible values above 6.5L)

In [ ]:
data_3=data_2[data_2['EngineV']<6.5]

In [ ]:
sns.distplot(data_2['Mileage'])

**Year**

In [ ]:
sns.distplot(data_3['Year'])

In [ ]:
q=data_3['Year'].quantile(0.01)
data_4=data_3[data_3['Year']>q]

In [ ]:
sns.distplot(data_4['Year'])

## 5. Final cleaned dataset

In [ ]:
data_cleaned=data_4.reset_index(drop=True)

## 6. Check linearity (OLS assumption)
Linear regression assumes a roughly linear relationship between each predictor and the target.

In [ ]:
f,(ax1,ax2,ax3)=plt.subplots(1,3,sharey=True,figsize=(25,3))
ax1.scatter(data_cleaned['Year'],data_cleaned['Price'])
ax1.set_title("price & year")
ax2.scatter(data_cleaned['EngineV'],data_cleaned['Price'])#ols assumption
ax2.set_title("price & enginev")
ax3.scatter(data_cleaned['Mileage'],data_cleaned['Price'])
ax3.set_title("price & Mileage")
plt.show()

## 7. Log-transform the target
`Price` is right-skewed and its relationship with predictors is closer to exponential than linear. Using `log(Price)` as the target straightens these relationships out, which better satisfies the linearity assumption.

In [ ]:
log_price = np.log(data_cleaned['Price'])

# Then we add it to our data frame
data_cleaned['log_price'] = log_price
data_cleaned

In [ ]:
f,(ax1,ax2,ax3)=plt.subplots(1,3,sharey=True,figsize=(25,3))
ax1.scatter(data_cleaned['Year'],data_cleaned['log_price'])
ax1.set_title("log_price & year")
ax2.scatter(data_cleaned['EngineV'],data_cleaned['log_price'])#ols assumption
ax2.set_title("log_price & enginev")
ax3.scatter(data_cleaned['Mileage'],data_cleaned['log_price'])
ax3.set_title("log_price & Mileage")
plt.show()

## 8. Check multicollinearity (VIF)
Variance Inflation Factor flags predictors that are highly correlated with each other (a problem for linear regression coefficient interpretation).

In [ ]:
data_cleaned.columns.values

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
variables=data_cleaned[['Mileage','Year','EngineV']]
vif=pd.DataFrame()
vif['VIF']=[variance_inflation_factor(variables.values,i) for i in range(variables.shape[1])]
vif

`Year` has the highest VIF and is dropped to reduce multicollinearity.

In [ ]:
data_no_multicollinearty=data_cleaned.drop(['Year'],axis=1)

## 9. Encode categorical variables
One-hot encode `Brand`, `Body`, `Engine Type`, and `Registration` (dropping first category to avoid the dummy-variable trap), then reorder columns for readability.

In [ ]:
data_with_dummies=pd.get_dummies(data_no_multicollinearty,drop_first=True)

In [ ]:
data_with_dummies.columns.values

In [ ]:
cols=['Price', 'Mileage', 'EngineV', 'log_price', 'Brand_BMW',
       'Brand_Mercedes-Benz', 'Brand_Mitsubishi', 'Brand_Renault',
       'Brand_Toyota', 'Brand_Volkswagen', 'Body_hatch', 'Body_other',
       'Body_sedan', 'Body_vagon', 'Body_van', 'Engine Type_Gas',
       'Engine Type_Other', 'Engine Type_Petrol', 'Registration_yes']

In [ ]:
data_preprocessed=data_with_dummies[cols]

## 10. Train/test split and feature scaling
Standardize the inputs and hold out 20% of the data for testing.

In [ ]:
target=data_preprocessed['log_price']
input=data_preprocessed.drop(['log_price'],axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(input)

In [ ]:
input_scaled=scaler.transform(input)


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(input_scaled,target,test_size=0.2,random_state=69)
                                               

## 11. Train the linear regression model

In [ ]:
reg=LinearRegression()
reg.fit(x_train,y_train)

In [ ]:
y_hat=reg.predict(x_train)

Predicted vs. actual values on the training set:

In [ ]:
plt.scatter(y_train,y_hat)
plt.xlabel("target(y_train)",size=18)
plt.ylabel('Predictions (y_hat)',size=18)
plt.xlim(6,13)
plt.ylim(6,13)
plt.show()
           

Residuals should be roughly normally distributed around zero:

In [ ]:
sns.distplot(y_train - y_hat)

# Include a title
plt.title("Residuals PDF", size=18)

## 12. Model performance and feature weights

In [ ]:
reg.score(x_train,y_train)

In [ ]:
reg.intercept_

In [ ]:
reg.coef_

In [ ]:
reg_summary=pd.DataFrame(input.columns.values,columns=['Features'])
reg_summary['weights']=reg.coef_
reg_summary

## 13. Evaluate on the test set

In [ ]:
#testing
data_cleaned["Brand"].unique()

In [ ]:
y_hat_test=reg.predict(x_test)

In [ ]:
plt.scatter(y_test, y_hat_test, alpha=0.2)
plt.xlabel('Targets (y_test)',size=18)
plt.ylabel('Predictions (y_hat_test)',size=18)
plt.xlim(6,13)
plt.ylim(6,13)
plt.show()

Convert log-price predictions back to actual price (€) and compare against the true targets:

In [ ]:
df_pf=pd.DataFrame(np.exp(y_hat_test),columns=['prediction'])
df_pf.head()

In [ ]:
df_pf['target']=np.exp(y_test)

In [ ]:
df_pf

In [ ]:
y_test=y_test.reset_index(drop=True)
y_test.head()

In [ ]:
df_pf['residual']=df_pf['target']-df_pf['prediction']

In [ ]:
df_pf['diff%']=np.absolute(df_pf['residual']/df_pf['target']*100)

Sorted by residual %, smallest to largest, to spot-check where the model does best/worst:

In [ ]:
pd.options.display.max_rows = 999
# Moreover, to make the dataset clear, we can display the result with only 2 digits after the dot 
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Finally, we sort by difference in % and manually check the model
df_pf.sort_values(by=['diff%'])

## Conclusion
The model explains a solid share of price variance using only Year, Mileage, EngineV, Brand, Body type, Engine type, and Registration status, after addressing skew (log target), outliers, and multicollinearity. Residual analysis on the test set shows most predictions within a reasonable percentage of actual price, with the largest errors concentrated in a handful of listings.